In [33]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from PIL import Image
import json
import faiss
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
import albumentations as A
from albumentations.pytorch import ToTensorV2
from transformers import ViTForImageClassification
from sklearn.preprocessing import LabelEncoder

# Configuration
DEVICE = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

Using device: mps


## Load data and model

In [34]:
from dataset import CarDamageDataset

# Load prepared data
data = torch.load("../../../development/models/quality_assurance/database/data_preparation_outputs.pth", weights_only=False)
label_encoder = data['label_encoder']
class_mapping = data['class_mapping']
severity_mapping = data['severity_mapping']

# Load model
model = ViTForImageClassification.from_pretrained(
    "google/vit-base-patch16-224-in21k",
    num_labels=len(label_encoder.classes_),
    ignore_mismatched_sizes=True
)
model.load_state_dict(torch.load("../../../development/models/quality_assurance/models/vit_damage_classifier.pth", map_location=DEVICE))
model = model.to(DEVICE)
model.eval()

print("Model loaded and ready for inference.")

Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch16-224-in21k and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model loaded and ready for inference.


## Feature Extractor model

In [35]:
class FeatureExtractor(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.vit = model.vit

    def forward(self, x):
        outputs = self.vit(pixel_values=x)
        return outputs.last_hidden_state[:, 0, :]  # Use the [CLS] token representation

feature_extractor_model = FeatureExtractor(model).to(DEVICE)
feature_extractor_model.eval()

print("Feature extractor model ready for inference.")

Feature extractor model ready for inference.


## Load all images for embedding

In [36]:
# CELL 4: Load All Images for Embedding
df_all_images = pd.read_csv('clean_image_dataset.csv') if Path('clean_image_dataset.csv').exists() else None

if df_all_images is None:
    # Reload data if needed
    from sklearn.model_selection import train_test_split
    df_damage = pd.read_csv("../../../development/database/cars_damage_dataset/data_damage.csv")
    df_damage = df_damage.drop(columns=['Unnamed: 0'])
    df_damage['image_path'] = df_damage['image'].apply(lambda x: f"../../../development/database/cars_damage_dataset/{x}")
    
    training_dir = Path("../../../development/database/cars_damage_dataset/cars_image_crash_dataset_addition/training")
    validation_dir = Path("../../../development/database/cars_damage_dataset/cars_image_crash_dataset_addition/validation")
    
    def load_additional_data(directory, source):
        data = []
        for class_dir in directory.iterdir():
            if class_dir.is_dir():
                for img_path in list(class_dir.glob("*.jpg")) + list(class_dir.glob("*.jpeg")):
                    data.append({
                        'image': str(img_path),
                        'classes': class_dir.name,
                        'source': source,
                        'image_path': str(img_path)
                    })
        return pd.DataFrame(data)
    
    df_train_add = load_additional_data(training_dir, 'additional_train')
    df_val_add = load_additional_data(validation_dir, 'additional_val')

    # ensure 'source' is exist and if no available, fill with raw_damage_dataset
    if 'source' not in df_damage.columns:
        df_damage['source'] = 'raw_damage_dataset'

    # Combine all dataframes
    df_all_images = pd.concat([
        df_damage[['image', 'classes', 'source', 'image_path']],
        df_train_add,
        df_val_add
    ], ignore_index=True)
    
    df_all_images['classes'] = df_all_images['classes'].str.lower()
    df_all_images['classes'] = df_all_images['classes'].replace({
        'unknown': 'no_damage',
        'unkown': 'no_damage',
        'no damage': 'no_damage'
    })
    df_all_images.to_csv('clean_image_dataset.csv', index=False)

print(f"Total images for embedding: {len(df_all_images)}")

Total images for embedding: 2143


In [37]:
# Define transform
val_transform = A.Compose([
    A.Resize(224, 224),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

# Extract embedding

In [38]:
def extract_embeddings(image_paths, batch_size=64):
    """
    Extract embeddings from images with purpose of creating 
    a feature vector for each image using the ViT model.
    """
    embeddings = []
    metadata = []

    with torch.no_grad():
        for i in range(0, len(image_paths), batch_size):
            batch_paths = image_paths[i:i + batch_size]

            batch_images = []
            valid_paths = []

            for img_path in batch_paths:
                try:
                    img = Image.open(img_path).convert("RGB")
                    img = np.array(img)
                    transformed = val_transform(image=img)
                    batch_images.append(transformed["image"])
                    valid_paths.append(img_path)
                except Exception as e:
                    print(f"Error loading {img_path}: {e}")

            if not batch_images:
                continue

            batch_images = torch.stack(batch_images).to(DEVICE)

            outputs = feature_extractor_model.vit(pixel_values=batch_images)

            # CLS token embedding
            batch_embeddings = outputs.last_hidden_state[:, 0, :]

            for path in valid_paths:
                row = df_all_images[df_all_images["image_path"] == path]
                img_class = row["classes"].iloc[0] if not row.empty else "unknown"

                metadata.append({
                    "image_path": path,
                    "class": img_class,
                    "severity": severity_mapping.get(img_class, 0),
                })

            embeddings.append(batch_embeddings.cpu().numpy())

    embeddings = np.vstack(embeddings)
    return embeddings, metadata

print("Extracting embeddings...")
embeddings, metadata = extract_embeddings(df_all_images["image_path"].values)
print(f"Extracted {embeddings.shape[0]} embeddings of dimension {embeddings.shape[1]}")

Extracting embeddings...
Extracted 2143 embeddings of dimension 768


In [39]:
# Save embeddings and metadata
np.save("embeddings.npy", embeddings)
pd.DataFrame(metadata).to_parquet("image_metadata.parquet")
print("Embeddings and metadata saved successfully.")

Embeddings and metadata saved successfully.


## Build FAISS Index

In [40]:
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings.astype('float32'))

# Save index
faiss.write_index(index, "faiss_cars_damage.index")
print(f"FAISS index built and saved with {index.ntotal} vectors of dimension {dimension}.")

FAISS index built and saved with 2143 vectors of dimension 768.


## Similarity Search Function

In [41]:
def find_similar_cars(image_path, k=5, min_severity=0):
    img = Image.open(image_path).convert("RGB")
    img = np.array(img)
    transformed = val_transform(image=img)
    query_image = transformed["image"].unsqueeze(0).to(DEVICE)

    # Extract query embedding
    with torch.no_grad():
        query_embedding = feature_extractor_model(query_image).cpu().numpy()

    # Search in FAISS index
    distances, indices = index.search(query_embedding.astype('float32'), k*2)

    # Get results
    similar_cars = []
    for idx, dist in zip(indices[0], distances[0]):
        if idx < len(metadata):
            meta = metadata[idx]
            if meta['severity'] >= min_severity:
                similar_cars.append({
                    "image_path": meta['image_path'],
                    "class": meta['class'],
                    "severity": meta['severity'],
                    "distance": dist
                })

    return similar_cars[:k] # Return top k results after filtering by severity

# Test the recommendation system
test_image = df_all_images['image_path'].iloc[0]
similar = find_similar_cars(test_image, k=5)
print(f"Similar cars to {test_image}:")

for idx, car in enumerate(similar, 1):
    print(f"{idx}. Path: {car['image_path']}, Class: {car['class']}, Severity: {car['severity']}, Distance: {car['distance']:.4f}")

Similar cars to ../../../development/database/cars_damage_dataset/image/0.jpeg:
1. Path: ../../../development/database/cars_damage_dataset/image/0.jpeg, Class: no_damage, Severity: 0, Distance: 0.0000
2. Path: ../../../development/database/cars_damage_dataset/image/483.jpeg, Class: no_damage, Severity: 0, Distance: 5.5705
3. Path: ../../../development/database/cars_damage_dataset/image/322.jpeg, Class: no_damage, Severity: 0, Distance: 5.7577
4. Path: ../../../development/database/cars_damage_dataset/image/1051.jpeg, Class: no_damage, Severity: 0, Distance: 5.7690
5. Path: ../../../development/database/cars_damage_dataset/image/1342.jpeg, Class: no_damage, Severity: 0, Distance: 5.8616


## Load car sales data

In [42]:
# CELL 10: Load Car Sales Data
df_car = pd.read_parquet("../../../development/database/car_sales_prediction_sales.parquet")
df_car = df_car[['car_id', 'date', 'customer_name', 'dealer_name', 'company', 'model', 'discount']]
print(f"Car sales data loaded: {len(df_car)} records")

Car sales data loaded: 23905 records


## Hybrid Recommendation System

In [48]:
def recommend_cars(image_path, df_car, budget_limit=500000000, income_level='medium', body_style=None, k=5):
    """Get similar cars based on damage"""
    similar_cars = find_similar_cars(image_path, k=k*2)

    # Get car details from sales data
    recommend = []
    for car in similar_cars:
        car_sample = df_car.sample(1).iloc[0]  # Randomly sample a car from sales data

        recommend.append({
            'car_id': car_sample['car_id'],
            'company': car_sample['company'],
            'model': car_sample['model'],
            'discount': car_sample['discount'],
            'damage_class': car['class'],
            'severity': car['severity'],
            'similarity_score': 1 - min(car['distance'], 1)
        })

    # Sort by similarity score
    recommend = sorted(recommend, key=lambda x: x['similarity_score'], reverse=True)
    return pd.DataFrame(recommend)

# Test the recommendation system with car sales data
sample_image = df_all_images['image_path'].iloc[0]
recommendations = recommend_cars(sample_image, df_car, budget_limit=500000000, income_level='medium', k=5)
print("\nRecommended cars based on damage similarity:")
print(recommendations)


Recommended cars based on damage similarity:
         car_id     company           model  discount damage_class  severity  \
0  C_CND_016869   Chevrolet        Corvette      0.30    no_damage         0   
1  C_CND_023362       Acura              RL      0.02    no_damage         0   
2  C_CND_012313  Mercedes-B         S-Class      0.02    no_damage         0   
3  C_CND_009595   Chevrolet          Malibu      0.30    no_damage         0   
4  C_CND_005986      Subaru         Outback      0.05    no_damage         0   
5  C_CND_021627       Dodge         Durango      0.02    03-severe         0   
6  C_CND_021041       Dodge      Ram Pickup      0.02    no_damage         0   
7  C_CND_020325    Chrysler  Town & Country      0.10    03-severe         0   
8  C_CND_017526    Plymouth         Voyager      0.30    no_damage         0   
9  C_CND_022337   Chevrolet     Monte Carlo      0.02    03-severe         0   

   similarity_score  
0               1.0  
1               0.0  
2      

## Save recommendation artifacts

In [49]:
# CELL 12: Save Recommendation Artifacts
# Save metadata for production use
import joblib

# Save label encoder and mappings
joblib.dump(label_encoder, 'label_encoder.pkl')
with open('severity_mapping.json', 'w') as f:
    json.dump(severity_mapping, f, indent=2)

# Save recommendation metadata
recommendation_metadata = {
    'model_name': 'vit_damage_classifier',
    'embedding_dim': dimension,
    'num_embeddings': len(metadata),
    'num_classes': len(label_encoder.classes_),
    'classes': label_encoder.classes_.tolist()
}

with open('recommendation_metadata.json', 'w') as f:
    json.dump(recommendation_metadata, f, indent=2)

print("Recommendation artifacts saved successfully!")

Recommendation artifacts saved successfully!
